In [1]:
import os
import sys
import torch
import numpy as np
import gymnasium as gym
import random
import matplotlib.pylab as plt
import time
SCRIPT_DIR = os.path.dirname(os.path.abspath("..src"))
sys.path.append(os.path.dirname(SCRIPT_DIR))
from src.wrappers.env_wrappers import WallObs, WindowViewObs, ObsToImage, MultiChannel
from src.wrappers.monitor_wrappers import BinaryMonitor, RoomMonitor

In [7]:
device = "mps:0"
grid_size = [10, 10]
eps_decay = 0.0001
n_mdp_actions = 6
window_size = 11
dryness = 0.05
gamma = 0.99
q_lr = 0.0001
r_lr = 0.0001

n_rows, n_colums = grid_size
# main_dir = "../general_models/Plants/reward_model/{}_{}/dry_{}/room/q_lr_{}/reward_lr_{}".format(n_rows, n_colums, dryness, q_lr, r_lr)
# main_dir = "../general_models/Plants/reward_model/{}_{}_channel/dry_{}/window_{}/eps_decay_{}/q_lr_{}/reward_lr_{}".format(n_rows, n_colums, dryness, window_size, eps_decay, q_lr, r_lr)

env = gym.make("gym_monitor/Plants-Watering-v1",
    grid_size=grid_size,
    n_plants=8,
    plants_dryness_prob=dryness,
    dry_difference=0.5,
    agent_start_pos=None,
    max_episode_steps=100,
    render_mode = "human",
    add_new_plants=False,
    add_more_plants=False,
    )

env = WallObs(env, grid_size=grid_size, n_walls=10)
env = WindowViewObs(env, window_size=window_size)
env = MultiChannel(env)
# env = ObsToImage(env)
# env = BinaryMonitor(env, full_monitor=False, monitor_cost=0.2)
env = RoomMonitor(env, full_monitor=False, monitor_cost=0.0, monitor_column_ind=5)

def select_action(q_values) -> int:
    """Choose an action from a Q value distribution/ break ties to a random action"""
    indices = np.argwhere(q_values == np.max(q_values))
    return random.choice(indices)[0]

In [8]:
n_checks = 23
n_ep = 1
n_seeds = 10

main_dir = "../general_models/Plants/reward_model/{}_{}_channel/dry_{}/room/window_{}/eps_decay_{}/q_lr_{}/reward_lr_{}".format(
    n_rows, n_colums, dryness, window_size, eps_decay, q_lr, r_lr)

# seeds_monitor_frequency = -1 * np.ones((n_seeds, n_checks, n_ep))
obs_env = -1 * np.ones((n_seeds, n_checks, n_ep, 6 , n_rows + 1, n_colums + 1))
obs_mon = -1 * np.ones((n_seeds, n_checks, n_ep))
for seed in range(n_seeds):
    for check in range(n_checks):
        q_model = torch.load(main_dir + "/checkpoints_{}/q_network_{}".format(check, seed), map_location=torch.device(device), weights_only=False)
        for episode in range(n_ep):
            ask_action = 0
            s, _ = env.reset()
            total_r, timestep, spill, cactus, water_plant = 0, 0, 0, 0, 0
            done = False
            actions, mdp_s, mon_s = [], [], []
            while timestep < 100 and not done:
                mdp_obs = torch.tensor(s["mdp"], dtype=torch.float32, device=device).unsqueeze(0)
                mon_obs = torch.tensor([s["monitor"]], dtype=torch.float32, device=device).unsqueeze(0)
                
                with torch.no_grad():
                    q_values = q_model[5:](torch.concat((q_model[:5](mdp_obs), mon_obs), 1))
                obs_env[seed, check, episode] = s["mdp"]
                obs_mon[seed, check, episode] = s["monitor"]
                # mdp_obs = torch.tensor(s["mdp"], dtype=torch.float32, device=device).unsqueeze(0)
                mdp_s.append(mdp_obs)
                # with torch.no_grad():
                #     q_values = q_model(mdp_obs)
                action = select_action(q_values.cpu().numpy().squeeze())
                actions.append(action)
                s, r, done, _, info = env.step({"mdp": action, "monitor": 0})
                # s, r, done, _, info = env.step({"mdp": action % n_mdp_actions, "monitor": action // n_mdp_actions})
                # if action // n_mdp_actions == 1:
                #     ask_action += 1
                reward = info["mdp_reward"] + r["monitor"]
                total_r += reward * (gamma**timestep)
                timestep += 1
            # seeds_monitor_frequency[seed, check, episode] = ask_action
env.close()
# np.save(main_dir + "/training_frequency_ask_moniotred", seeds_monitor_frequency)
np.save(main_dir + "/env_obs.npy", obs_env)
np.save(main_dir + "/monitor_obs.npy", obs_mon)

In [19]:
main_dir

'../general_models/Plants/reward_model/10_10_channel/dry_0.05/room/window_11/eps_decay_0.0001/q_lr_0.0001/reward_lr_0.0001'

In [ ]:
ep_reward = []
for i in range(10):
    s, _ = env.reset()
    total_r, timestep, spill, cactus, water_plant, water_diff_plant = 0, 0, 0, 0, 0, 0
    done = False
    actions, mdp_s, mon_s = [], [], []
    env.render()
    flag = False
    while timestep < 100:
        mdp_obs = torch.tensor(s["mdp"], dtype=torch.float32, device=device).unsqueeze(0)
        mon_obs = torch.tensor([s["monitor"]], dtype=torch.float32, device=device).unsqueeze(0)
        mdp_s.append(mdp_obs)
        mon_s.append(mon_obs)
        action = env.action_space.sample()["mdp"]
        actions.append(action)
        plant = env.get_grid()[1, env.get_agent_pos()[0], env.get_agent_pos()[1]]
        if plant == 1.0 and not flag:
            action = 4
            flag = True
        s, r, done, _, info = env.step({"mdp": action, "monitor": 0})
        if action == 4:
            if plant == 0.0:  # spill water in the floor
                spill += 1
            elif plant == 0.5: # water cactus
                cactus += 1
                print("r", info["mdp_reward"])
            elif plant == 1.0: # water cactus
                water_plant += 1
            else:
                water_diff_plant += 1
                print("r", info["mdp_reward"])
        reward = info["mdp_reward"] + r["monitor"]
        total_r += reward * (gamma**timestep)
        timestep += 1
        # env.render()
    ep_reward.append(total_r)
    print("spill", spill)
    print("cactus", cactus)
    print("water_plant", water_plant)
    print("water_diff_plant", water_diff_plant)